<center>
<img src="https://www.economicas.uba.ar/wp-content/uploads/2020/08/cropped-logo_FCE.png" width="200"/>
</center>

# Muestreo en Python

**Materia:** Métodos Predictivos para la Gestión  
**Tecnicatura en Gestión de Datos — FCE-UBA**  

---

Dataset utilizado: [Predict Students' Dropout and Academic Success](https://archive.ics.uci.edu/dataset/697/predict+students+dropout+and+academic+success)

In [ ]:
pip install ucimlrepo

In [ ]:
import pandas as pd
from ucimlrepo import fetch_ucirepo
import numpy as np

In [ ]:
# 1. Traer la base de datos
dataset = fetch_ucirepo(id=697)
# https://archive.ics.uci.edu/dataset/697/predict+students+dropout+and+academic+success

# Combinamos features y target en un único DataFrame
df = pd.concat([dataset.data.features, dataset.data.targets], axis=1)

In [ ]:
df

In [ ]:
# imprimimos la dimensionalidad del archivo
df.shape

## Muestreo Aleatorio Simple (MAS)

**Muestreo Aleatorio Simple**

Obtener "n" observaciones de modo aleatorio.  
Puede realizarse con o sin reemplazo.

### MAS sin reposición

In [ ]:
# generamos una submuestra aleatoria con n=100 sin reposición y luego la imprimimos
muestra_MAS = df.sample(100, random_state=42)
muestra_MAS

In [ ]:
muestra_MAS.shape

### MAS con reposición

In [ ]:
# generamos una submuestra aleatoria con n=100 con reposición (replace=True)
muestra_MAS_crepo = df.sample(100, replace=True, random_state=42)
muestra_MAS_crepo

Se observan repetidos los índices 1346 y 3252.

In [ ]:
# verificamos los índices duplicados en la muestra con reposición
muestra_MAS_crepo[muestra_MAS_crepo.index.duplicated(keep=False)].sort_index()

## Muestreo por Porcentaje

In [ ]:
# generamos una submuestra aleatoria por porcentaje (frac=2.3%) sin reposición
muestra_porc = df.sample(frac=0.023, random_state=42)
muestra_porc.shape

In [ ]:
muestra_porc

## Muestreo con filtro sobre variable

In [ ]:
# Filtramos registros con Marital Status == 4 y tomamos una muestra aleatoria
muestra_FILT = df[df['Marital Status'] == 4].sample(15, random_state=42)
muestra_FILT

También se podría hacer con un valor mayor o menor al de una variable. Por ejemplo la edad, si tuviéramos el caso.

## Reducir el muestreo de la clase mayoritaria

Cuando existe desbalance de clases, podemos **submuestrear la clase mayoritaria** para equipararla con la minoritaria.  
Esto se conoce como **undersampling**.

In [ ]:
df.groupby('Daytime/evening attendance')['Previous qualification'].count()

In [ ]:
# se toman muestras de los casos según el valor que toma cada clase
min_size = df['Daytime/evening attendance'].value_counts().min()
muestra_under = df.groupby('Daytime/evening attendance').apply(
    lambda x: x.sample(min_size, random_state=42)
).reset_index(drop=True)

In [ ]:
muestra_under.groupby('Daytime/evening attendance')['Previous qualification'].count()

## Muestreo Sistemático

In [ ]:
n = 100  # cantidad de observaciones que queremos en la muestra
N = len(df)  # tamaño de la población
k = N // n  # división entera (paso de selección)
start = np.random.randint(0, k)
indices = range(start, N, k)
muestra_sistem = df.iloc[indices]
muestra_sistem

## Muestreo por Conglomerados

In [ ]:
# Muestreo por conglomerados usando la variable Course como variable de agrupación
np.random.seed(42)
clusters_unicos = df['Course'].unique()
n_clusters = 4
clusters_seleccionados = np.random.choice(clusters_unicos, size=n_clusters, replace=False)
print(f"Clusters seleccionados: {clusters_seleccionados}")
muestra_cluster = (
    df[df['Course'].isin(clusters_seleccionados)]
    .groupby('Course')
    .apply(lambda x: x.sample(25, random_state=42))
    .reset_index(drop=True)
)
print(f"Tamaño muestra: {len(muestra_cluster)}")
muestra_cluster.head()

In [ ]:
muestra_cluster.groupby('Course')['Previous qualification'].count()

## Muestreo Estratificado

Hagamos un estratificado por género

In [ ]:
n = 100  # tamaño deseado
# Calculamos proporción de Daytime/evening attendance (proxy de género)
proporciones = df['Daytime/evening attendance'].value_counts(normalize=True)
proporciones

In [ ]:
# muestreo dentro de cada estrato
muestra = []
for estrato, prop in proporciones.items():
    n_estrato = round(n * prop)
    muestra_estrato = df[df['Daytime/evening attendance'] == estrato].sample(n_estrato, random_state=42)
    muestra.append(muestra_estrato)
muestra_estratificada = pd.concat(muestra)

In [ ]:
# Crear un DataFrame con ambas distribuciones
comparacion = pd.DataFrame({
    'Poblacion': df['Target'].value_counts(normalize=True).round(4),
    'Muestra por estrato': muestra_estratificada['Target'].value_counts(normalize=True).round(4)
})
comparacion

## Comparación de distribución del Target

In [ ]:
tabla_final = pd.DataFrame({
    'Población': df['Target'].value_counts(normalize=True).round(4),
    'Estratificado_Daytime': muestra_estratificada['Target'].value_counts(normalize=True).round(4)
}).T
tabla_final

In [ ]:
# Lista de datasets
datasets = {
    "Población": df,
    "MAS": muestra_MAS,
    "MAS c/reposición": muestra_MAS_crepo,
    "Porcentaje": muestra_porc,
    "Sistemático": muestra_sistem,
    "Conglomerado": muestra_cluster,
    "Estratificado": muestra_estratificada,
}
variable = "Curricular units 1st sem (grade)"

tabla = pd.DataFrame([
    {
        "Método": nombre,
        "Media": ds[variable].mean(),
        "Desvío": ds[variable].std(),
        "Mínimo": ds[variable].min(),
        "Máximo": ds[variable].max(),
    }
    for nombre, ds in datasets.items()
])
tabla = tabla.round(2)
tabla

## Comparación gráfica entre métodos de muestreo

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Variable a analizar
variable = "Curricular units 1st sem (grade)"

# Unificar datasets
dataframes = []

datasets = {
    "Población": df,
    "MAS": muestra_MAS,
    "MAS c/reposición": muestra_MAS_crepo,
    "Porcentaje": muestra_porc,
    "Sistemático": muestra_sistem,
    "Conglomerado": muestra_cluster,
    "Estratificado": muestra_estratificada,
}

for nombre, ds in datasets.items():
    temp = ds[[variable]].copy()
    temp["Metodo"] = nombre
    dataframes.append(temp)

df_plot = pd.concat(dataframes)

# Eliminamos NA
df_plot = df_plot.dropna()

# Boxplot
plt.figure(figsize=(12, 6))
grupos = [df_plot[df_plot["Metodo"] == m][variable].values for m in datasets.keys()]
plt.boxplot(grupos, labels=datasets.keys())
plt.title(f"Comparación de métodos de muestreo — {variable}")
plt.xlabel("Método de muestreo")
plt.ylabel("Nota")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## Otra Propuesta

Estratificado por la variable `Course`

In [ ]:
n = 200  # tamaño deseado

# Calculamos proporción de Curso
proporciones_curso = df['Course'].value_counts(normalize=True)

# Muestreamos dentro de cada estrato
muestra = []
for curso, prop in proporciones_curso.items():
    n_estrato = round(n * prop)
    if n_estrato > 0:
        disponibles = df[df['Course'] == curso]
        n_real = min(n_estrato, len(disponibles))
        muestra_estrato = disponibles.sample(n_real, random_state=42)
        muestra.append(muestra_estrato)

muestra_estratificada_course = pd.concat(muestra)

In [ ]:
# Crear un DataFrame con ambas distribuciones
comparacion = pd.DataFrame({
    'Poblacion': proporciones_curso.round(4),
    'Muestra por estrato': muestra_estratificada_course['Course'].value_counts(normalize=True).round(3)
})
comparacion = comparacion.round(4)
comparacion

In [ ]:
# Crear un DataFrame con ambas distribuciones
comparacion = pd.DataFrame({
    'Población': df['Target'].value_counts(normalize=True).round(4),
    'Estratificado_Course': muestra_estratificada_course['Target'].value_counts(normalize=True).round(4)
}).T
comparacion = comparacion.round(4)
comparacion